# 第 45 章：Capstone 学生 Handout 与助教评分指南

这个 notebook 用一个极小的 handout / TA-guide release table，对比“只按发布优先级排序”的 naive baseline 和“先检查政策、助教校准、学生清晰度与样例质量”的发布前 workflow。

In [ ]:
from __future__ import annotations

import csv
import platform
from collections import Counter
from pathlib import Path

DATA_PATH = Path('../../data/small/capstone_student_handout_ta_guide_demo.csv')


def load_items(path):
    rows = []
    with path.open(encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            row['release_priority_score'] = float(row['release_priority_score'])
            rows.append(row)
    return rows


def ordered_counts(rows, field):
    counts = Counter(row[field] for row in rows)
    return {key: counts[key] for key in sorted(counts)}


rows = load_items(DATA_PATH)
print(f'Loaded {len(rows)} handout / TA-guide items from {DATA_PATH.name}')
print('Python version:', platform.python_version())
print('Audience:', ordered_counts(rows, 'audience'))
print('Policy alignment:', ordered_counts(rows, 'policy_alignment_status'))
print('TA guidance:', ordered_counts(rows, 'ta_guidance_status'))
print('Reference route:', ordered_counts(rows, 'reference_route'))


In [ ]:
reference_ready_ids = sorted(
    row['item_id']
    for row in rows
    if row['reference_route'] == 'ready_to_release'
)
top4_priority = sorted(
    rows,
    key=lambda row: row['release_priority_score'],
    reverse=True,
)[:4]
baseline_ready_hits = sum(
    row['reference_route'] == 'ready_to_release'
    for row in top4_priority
)
baseline_policy_hits = sum(
    row['reference_route'] == 'policy_review'
    for row in top4_priority
)
baseline_ready_precision = baseline_ready_hits / len(top4_priority)
baseline_policy_intrusion = baseline_policy_hits / len(top4_priority)

print('Reference ready-to-release items:', reference_ready_ids)
print('Top-4 by release priority:')
for row in top4_priority:
    print(
        f"  {row['item_id']}: priority={row['release_priority_score']:.2f}, "
        f"route={row['reference_route']}, section={row['package_section']}"
    )
print('Baseline release precision:', round(baseline_ready_precision, 3))
print('Baseline policy-review intrusion:', round(baseline_policy_intrusion, 3))


In [ ]:
def route_release_item(row):
    if row['policy_alignment_status'] != 'aligned':
        return 'policy_review', 'policy, privacy, AI-use, or grading rule still needs review'
    if row['audience'] in {'ta', 'both'} and row['ta_guidance_status'] != 'clear':
        return 'ta_calibration_needed', 'TA guidance, anchors, examples, or escalation path is not stable'
    if row['student_clarity_status'] != 'clear':
        return 'revise_before_release', 'student-facing task wording is not yet clear enough'
    if row['example_quality_status'] in {'weak', 'partial'}:
        return 'revise_before_release', 'examples are too weak or incomplete for release'
    return 'ready_to_release', 'section is aligned, clear, and supported by usable examples'


routed_rows = []
for row in rows:
    route, reason = route_release_item(row)
    routed = dict(row)
    routed['route'] = route
    routed['reason'] = reason
    routed_rows.append(routed)

print('Release workflow routes:')
for row in routed_rows:
    print(
        f"  {row['item_id']}: route={row['route']}, "
        f"reference={row['reference_route']}, reason={row['reason']}"
    )


In [ ]:
ready_queue = [row for row in routed_rows if row['route'] == 'ready_to_release']
revise_queue = [row for row in routed_rows if row['route'] == 'revise_before_release']
policy_queue = [row for row in routed_rows if row['route'] == 'policy_review']
ta_queue = [row for row in routed_rows if row['route'] == 'ta_calibration_needed']

workflow_accuracy = sum(
    row['route'] == row['reference_route']
    for row in routed_rows
) / len(routed_rows)
ready_precision = sum(
    row['reference_route'] == 'ready_to_release'
    for row in ready_queue
) / max(len(ready_queue), 1)

print('Ready-to-release queue:', [row['item_id'] for row in ready_queue])
print('Revise-before-release queue:', [row['item_id'] for row in revise_queue])
print('Policy-review queue:', [row['item_id'] for row in policy_queue])
print('TA-calibration queue:', [row['item_id'] for row in ta_queue])
print('Workflow route accuracy:', round(workflow_accuracy, 3))
print('Structured ready precision:', round(ready_precision, 3))


In [ ]:
def release_revision_steps(row):
    steps = []
    if row['policy_alignment_status'] != 'aligned':
        steps.append('先由课程团队复核政策、隐私、AI 使用或评分规则，暂不发布。')
    if row['audience'] in {'ta', 'both'} and row['ta_guidance_status'] != 'clear':
        steps.append('补 TA anchors、示范反馈和无法判断时的升级路径。')
    if row['student_clarity_status'] != 'clear':
        steps.append('重写学生任务说明，让下一步动作、交付物和截止时间更清楚。')
    if row['example_quality_status'] == 'weak':
        steps.append('新增强样例和反例，不要只给抽象规则。')
    elif row['example_quality_status'] == 'partial':
        steps.append('补完整样例，确保学生或助教能照着执行。')
    return steps or ['可以进入发布包；发布后仍需记录学生和助教反馈。']


for row in routed_rows:
    if row['route'] != 'ready_to_release':
        print(f"\n{row['item_id']} -> {row['route']} ({row['package_section']})")
        for step in release_revision_steps(row):
            print(' -', step)


In [ ]:
student_handout_outline = [
    'Project goal and success criterion',
    'Timeline and checkpoints',
    'Data, privacy, and sharing boundary',
    'Notebook reproducibility requirements',
    'Rubric gates and minimum evidence lines',
    'AI-use policy and disclosure examples',
    'Help path, office hours, and human signoff',
    'Final submission checklist',
]
ta_guide_outline = [
    'Rubric anchor table',
    'Grade norming cases',
    'Feedback examples',
    'AI-use statement review examples',
    'Escalation path for ambiguous cases',
    'Post-grading audit checklist',
]

release_assistant_prompt = '''你是我的 capstone 材料发布助手。

任务：
1. 先阅读 handout / TA-guide release table，不要只按发布优先级排序；
2. 先检查 policy alignment；
3. 再检查 TA guidance、student clarity 和 example quality；
4. 把每个模块 route 到 ready_to_release、revise_before_release、
   policy_review 或 ta_calibration_needed；
5. 对每个非 ready 模块输出发布前修订 checklist。

输出格式：
- Ready-to-release modules
- Revise-before-release modules
- Policy-review modules
- TA-calibration modules
- Release checklist by module
'''

print('Student handout outline:')
for item in student_handout_outline:
    print(' -', item)

print('\nTA guide outline:')
for item in ta_guide_outline:
    print(' -', item)

print('\nAssistant prompt:')
print(release_assistant_prompt)


In [ ]:
release_snapshot = {
    'dataset': DATA_PATH.name,
    'n_release_items': len(rows),
    'baseline_top4_ready_precision': round(baseline_ready_precision, 3),
    'baseline_policy_intrusion': round(baseline_policy_intrusion, 3),
    'workflow_route_accuracy': round(workflow_accuracy, 3),
    'ready_to_release': [row['item_id'] for row in ready_queue],
    'policy_review': [row['item_id'] for row in policy_queue],
    'ta_calibration_needed': [row['item_id'] for row in ta_queue],
    'python_version': platform.python_version(),
}

print('Release snapshot:')
for key, value in release_snapshot.items():
    print(f'  {key}: {value}')
